### AUTOCORRECTOR BASE + CONTEXTO

#### INDICE

- [Procedimiento basico](#Procedimiento-basico)
- [Evaluación de mejores hiperparametros](#Evaluación-de-mejores-hiperparametros)
- [Conclusiones](#Conclusiones)

### Procedimiento basico

<pre>
Autocorrector base + contexto (Procedimiento):
    1) Llega una frase a corregir.
    2) Para cada palabra de la frase:
        Si esta en el vocabulario => La dejamos igual
        Si no:
            Obtenemos varias palabras que están cerca, que son frecuentes y nos quedamos con la de mayor probabilidad
            de contexto.
    3) Se devuelve la frase corregida.
</pre>

### Evaluación de mejores hiperparametros

El autocorrector base + contexto usa 2 hiperpametros, el min_apperance, que es el número mínimo de apariciones
que debe tener una palabra para no ser considerada "unk"; y el tamaño de los ngramas.

En cuanto al min_apperance, vamos a probar todos los valores comprendidos entre [2, 20]. Sin embargo,
para el tamaño de los ngramas, solo va a ser de 2, es decir, no nos vamos a molestarnos en experimentar
con el porque en el caso de que sea 1, la información que aporta sobre el contexto es 0, y en el caso de
tener tamaño de ngrama > 2, no nos caben los datos en la memoria RAM.

In [1]:
from autocorrector import Autocorrector
from cargador_corpus import LectorCorpus

lector_corpus = LectorCorpus("../libros")
X_train = lector_corpus.get_corpus()
X_val = [
    ("el perro corria por el parque sin mirar atras", "el perro corria por el parqe sin mirar atras"),
    ("la niña jugaba tranquila en el jardin de su casa", "la niña jugaba tranquila en el jardn de su casa"),
    ("mañana iremos todos juntos a visitar a la abuela", "mañana iremos todos juntos a vsitar a la abuela"),
    ("el profesor explico la leccion con mucha paciencia", "el profesor explcio la leccion con mucha paciencia"),
    ("los amigos quedaron para cenar en un restaurante cercano", "los amigos quedaron para cenar en un restrante cercano"),
    ("la lluvia caia suavemente sobre el tejado de la casa", "la lluvia caia suavemente sobre el tejdao de la casa"),
    ("el tren llego puntual a la estacion principal del pueblo", "el tren llego puntual a la estcion principal del pueblo"),
    ("mi hermano pequeño rompio el juguete sin querer hacerlo", "mi hermano pequeño rompio el jugete sin querer hacerlo"),
    ("los niños aprendieron rapidamente a leer y escribir bien", "los niños aprendieron rapidamente a leer y escrbir bien"),
    ("el viento movia lentamente las hojas de los arboles", "el viento movia lentamente las hojas de los arbols"),
    ("la comida estaba deliciosa y todos repitieron varias veces", "la comida estaba delciosa y todos repitieron varias veces"),
    ("el coche azul estaba aparcado frente a la puerta principal", "el coche azul estaba aparcado frente a la pueta principal"),
    ("fuimos caminando despacio hasta llegar al final del sendero", "fuimos caminando despacio hasta llegar al fnal del sendero"),
    ("ella escribio una carta larga contando toda la verdad", "ella escribio una carta larga contando toda la vedad"),
    ("el gato dormia placidamente encima del sofa del salon", "el gato dormia placidamente encima del sfa del salon"),
    ("los estudiantes prepararon el examen durante toda la semana", "los estudiantes prepararon el exmen durante toda la semana"),
    ("la puerta vieja hacia ruido cada vez que alguien entraba", "la puerta vieja hacia ruido cada vez que algien entraba"),
    ("el jardin estaba lleno de flores de muchos colores distintos", "el jardin estaba lleno de flores de muchos colres distintos"),
    ("mi padre compro pan fresco en la tienda de la esquina", "mi padre compro pan fresco en la tenda de la esquina"),
    ("los vecinos hablaron durante horas sobre el problema comun", "los vecinos hablaron durante horas sobre el problma comun"),
]

normalizar = lambda s: s.lower().translate(str.maketrans("áéíóúü", "aeiouu"))


def corregir_frase_contexto(frase: str, X_train: list[str], numero_ngramas: int = 2, min_apperance: int = 10) -> str:
    autocorrector_contexto = Autocorrector(modo="contexto", numero_ngramas=numero_ngramas, min_apperance=min_apperance)
    autocorrector_contexto.fit(X_train)
    return autocorrector_contexto.corregir(frase)


def medir_accuracy_contexto(X_val: list, X_train: list[str], numero_ngramas: int = 2, min_apperance: int = 10, devolver_detalles: bool = True) -> dict:
    autocorrector_contexto = Autocorrector(modo="contexto", numero_ngramas=numero_ngramas, min_apperance=min_apperance)
    autocorrector_contexto.fit(X_train)

    total = len(X_val)
    aciertos = 0
    detalles = []

    for frase_correcta, frase_con_error in X_val:
        frase_correcta_norm = normalizar(frase_correcta)
        frase_con_error_norm = normalizar(frase_con_error)
        pred = autocorrector_contexto.corregir(frase_con_error)
        pred_norm = normalizar(pred)
        ok = pred_norm == frase_correcta_norm
        aciertos += int(ok)

        if devolver_detalles:
            detalles.append({
                "frase_correcta": frase_correcta_norm,
                "frase_con_error": frase_con_error_norm,
                "prediccion": pred_norm,
                "acierto": ok,
            })

    resultado = {"accuracy": aciertos / total if total > 0 else 0.0, "aciertos": aciertos, "total": total}

    if devolver_detalles:
        resultado["detalles"] = detalles

    return resultado


print("=================================================")
print("    Evaluación de hiperparámetros")
print("=================================================\n")

mejor = None
for min_apperance in range(2, 21):
    r = medir_accuracy_contexto(X_val, X_train, numero_ngramas=2, min_apperance=min_apperance, devolver_detalles=False)
    print(f"min_apperance={min_apperance} -> {r['accuracy']:.2%} ({r['aciertos']}/{r['total']})")
    if mejor is None or r['accuracy'] > mejor['resultado']['accuracy']:
        mejor = {"numero_ngramas": 2, "min_apperance": min_apperance, "resultado": r}

print()
print(
    f"Mejor -> numero_ngramas={mejor['numero_ngramas']}, min_apperance={mejor['min_apperance']} | "
    f"{mejor['resultado']['accuracy']:.2%} ({mejor['resultado']['aciertos']}/{mejor['resultado']['total']})"
)

res = medir_accuracy_contexto(X_val, X_train, numero_ngramas=mejor['numero_ngramas'], min_apperance=mejor['min_apperance'], devolver_detalles=False)
print(f"Seleccionado para detalle -> {res['accuracy']:.2%} ({res['aciertos']}/{res['total']})")
print()

    Evaluación de hiperparámetros

min_apperance=2 -> 75.00% (15/20)
min_apperance=3 -> 75.00% (15/20)
min_apperance=4 -> 70.00% (14/20)
min_apperance=5 -> 70.00% (14/20)
min_apperance=6 -> 75.00% (15/20)
min_apperance=7 -> 70.00% (14/20)
min_apperance=8 -> 70.00% (14/20)
min_apperance=9 -> 70.00% (14/20)
min_apperance=10 -> 75.00% (15/20)
min_apperance=11 -> 75.00% (15/20)
min_apperance=12 -> 70.00% (14/20)
min_apperance=13 -> 60.00% (12/20)
min_apperance=14 -> 60.00% (12/20)
min_apperance=15 -> 60.00% (12/20)
min_apperance=16 -> 60.00% (12/20)
min_apperance=17 -> 55.00% (11/20)
min_apperance=18 -> 55.00% (11/20)
min_apperance=19 -> 55.00% (11/20)
min_apperance=20 -> 50.00% (10/20)

Mejor -> numero_ngramas=2, min_apperance=2 | 75.00% (15/20)
Seleccionado para detalle -> 75.00% (15/20)



In [2]:
res_mejor = medir_accuracy_contexto(
    X_val,
    X_train,
    numero_ngramas=mejor['numero_ngramas'],
    min_apperance=mejor['min_apperance'],
)
print(
    f"Frases con la mejor combinacion de hiperparametros: numero_ngramas={mejor['numero_ngramas']}, "
    f"min_apperance={mejor['min_apperance']}"
)
print()

for i, item in enumerate(res_mejor.get("detalles", []), start=1):
    estado = "OK" if item["acierto"] else "FAIL"
    print(f"[{i:02d}] {estado}")
    print(f"  error:      {item['frase_con_error']}")
    print(f"  prediccion: {item['prediccion']}")
    print(f"  correcta:   {item['frase_correcta']}")
    print()

Frases con la mejor combinacion de hiperparametros: numero_ngramas=2, min_apperance=2

[01] FAIL
  error:      el perro corria por el parqe sin mirar atras
  prediccion: el perro corria por el parte sin mirar atras
  correcta:   el perro corria por el parque sin mirar atras

[02] OK
  error:      la niña jugaba tranquila en el jardn de su casa
  prediccion: la niña jugaba tranquila en el jardin de su casa
  correcta:   la niña jugaba tranquila en el jardin de su casa

[03] OK
  error:      mañana iremos todos juntos a vsitar a la abuela
  prediccion: mañana iremos todos juntos a visitar a la abuela
  correcta:   mañana iremos todos juntos a visitar a la abuela

[04] OK
  error:      el profesor explcio la leccion con mucha paciencia
  prediccion: el profesor explico la leccion con mucha paciencia
  correcta:   el profesor explico la leccion con mucha paciencia

[05] FAIL
  error:      los amigos quedaron para cenar en un restrante cercano
  prediccion: los amigos quedaron para cenar en

### Conclusiones

Como podemos observar el mejor accuracy obtenido es 75%, sin embargo, hay varios valores de min_apperance
para los que ocurre esto, por ello, escogemos el valor más alto de estos porque ayudará a que nuestro
autocorrector puede generalizar mucho mejor las palabras, así que el mejor es min_apperance = 11.